In [7]:
import pandas as pd

INPUT_PATH = "https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz"
OUTPUT_PATH = './data/openfoodfacts_clean.csv'

nutriscore_cols = [
    'energy_100g', 'fat_100g', 'saturated-fat_100g', 
    'sugars_100g', 'fiber_100g', 'proteins_100g', 'carbohydrates_100g',
    'salt_100g'
]
target = ['nutriscore_grade']
identity_cols = ['code', 'product_name']
cols = nutriscore_cols + target + identity_cols

def process_data(file_path, cols, chunk_size=50000):
    reader = pd.read_csv(
        file_path, compression='gzip', sep='\t',
        on_bad_lines='skip', chunksize=chunk_size,
        low_memory=False, usecols=cols
    )

    clean_chunks = []

    for chunk in reader:
        # 1. Suppression des Na dans Nutri-Score ET Product Name
        temp_chunk = chunk.dropna(subset=['nutriscore_grade', 'product_name']).copy()
        
        # 2. Nettoyage Nutri-Score
        temp_chunk['nutriscore_grade'] = temp_chunk['nutriscore_grade'].str.lower()
        temp_chunk = temp_chunk[temp_chunk['nutriscore_grade'].isin(['a', 'b', 'c', 'd', 'e'])]

        # 3. Conversion numérique (Energy_100g inclus)
        for col in nutriscore_cols:
            temp_chunk[col] = pd.to_numeric(temp_chunk[col], errors='coerce').fillna(0)

        # 4. Filtres Outliers
        for col in nutriscore_cols:
            if col != 'energy_100g':
                temp_chunk = temp_chunk[(temp_chunk[col] >= 0) & (temp_chunk[col] <= 100)]
        
        temp_chunk = temp_chunk[(temp_chunk['energy_100g'] >= 0) & (temp_chunk['energy_100g'] <= 3700)]

        # 5. Doublons
        temp_chunk = temp_chunk.drop_duplicates(subset=['code'])

        if not temp_chunk.empty:
            clean_chunks.append(temp_chunk)

    # 6. Concaténation et sauvegarde (Hors de la boucle)
    if clean_chunks:
        df = pd.concat(clean_chunks, ignore_index=True)
        df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
        print(f"Succès : {len(df)} lignes sauvegardées dans {OUTPUT_PATH}")
        return df
    return pd.DataFrame()



In [8]:
df_final = process_data(INPUT_PATH, cols)

Succès : 1331710 lignes sauvegardées dans ./data/openfoodfacts_clean.csv
